# Etapa 1 — Dados do HMDA 2025

Projeto SME0828 — Introdução à Ciência de Dados, ICMC/USP.

Este notebook baixa os arquivos LAR e TS, confere as contagens e salva os dados do LAR em Parquet.

### Bibliotecas

In [1]:
from pathlib import Path
import os

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents) if (path / "data").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Abra este notebook dentro do reposit?rio, que deve conter a pasta data/")
os.chdir(ROOT)

import os
import urllib.request
import zipfile
from datetime import date

import duckdb

O LAR reúne os pedidos de crédito. O TS lista as instituições e permite conferir os totais.

In [2]:
PASTA = "data"
os.makedirs(PASTA, exist_ok=True)

URL = "https://files.ffiec.cfpb.gov/static-data/snapshot/2025/"
arquivos = ["2025_public_lar_csv.zip", "2025_public_ts_csv.zip"]

for arquivo in arquivos:
    caminho = os.path.join(PASTA, arquivo)
    if not os.path.exists(caminho):
        print("Baixando", arquivo, "...")
        urllib.request.urlretrieve(URL + arquivo, caminho)
    print(arquivo, "-", round(os.path.getsize(caminho) / 1e6), "MB")

print("Data:", date.today())

2025_public_lar_csv.zip - 737 MB
2025_public_ts_csv.zip - 0 MB
Data: 2026-11-27


### Extração dos arquivos

In [3]:
zipfile.ZipFile(f"{PASTA}/2025_public_lar_csv.zip").extract("2025_public_lar_csv.csv", PASTA)
zipfile.ZipFile(f"{PASTA}/2025_public_ts_csv.zip").extract("2025_public_ts_csv.csv", PASTA)

lar_csv = f"{PASTA}/2025_public_lar_csv.csv"
ts_csv = f"{PASTA}/2025_public_ts_csv.csv"
print("Tamanho do CSV:", round(os.path.getsize(lar_csv) / 1e9, 1), "GB")

Tamanho do CSV: 5.1 GB


Comparamos as contagens de linhas, instituições e colunas com os valores esperados.

In [4]:
con = duckdb.connect()
con.execute(f"CREATE VIEW lar AS SELECT * FROM read_csv('{lar_csv}', all_varchar = true)")
con.execute(f"CREATE VIEW ts AS SELECT * FROM read_csv('{ts_csv}')")

linhas, instituicoes = con.execute("SELECT count(*), count(DISTINCT lei) FROM lar").fetchone()
colunas = len(con.execute("DESCRIBE lar").fetchall())
instituicoes_ts, linhas_ts = con.execute("SELECT count(*), sum(lar_count) FROM ts").fetchone()

print(f"Linhas:       {linhas:,}  (esperado pelo TS: {linhas_ts:,})")
print(f"Colunas:      {colunas}  (esperado: 99)")
print(f"Instituições: {instituicoes:,}  (esperado pelo TS: {instituicoes_ts:,})")

assert linhas == linhas_ts and colunas == 99 and instituicoes == instituicoes_ts, "Base incompleta!"
print("\nBase completa!")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Linhas:       13,543,606  (esperado pelo TS: 13,543,606)
Colunas:      99  (esperado: 99)
Instituições: 4,782  (esperado pelo TS: 4,782)

Base completa!


O Parquet ocupa menos espaço e é mais rápido de ler. Salvamos apenas as colunas usadas no projeto.

In [5]:
todas = [c[0] for c in con.execute("DESCRIBE lar").fetchall()]
manter = [c for c in todas if c[-2:] not in ("_2", "_3", "_4", "_5") and c != "multifamily_affordable_units"]
print("Colunas mantidas:", len(manter), "de", len(todas))

con.execute(f"COPY (SELECT {', '.join(manter)} FROM lar) TO '{PASTA}/hmda_2025.parquet' (FORMAT parquet, COMPRESSION zstd)")
con.close()

print("Parquet:", round(os.path.getsize(f"{PASTA}/hmda_2025.parquet") / 1e6), "MB")

Colunas mantidas: 75 de 99


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Parquet: 495 MB


Depois da conversão, removemos os arquivos CSV extraídos.

In [6]:
os.remove(lar_csv)
os.remove(ts_csv)